# Attention & Encoder-Decoder Models — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/notebook/attention-seq2seq/attention-seq2seq-lab.ipynb)

Companion notebook for the **Attention & Encoder-Decoder Models** track
(`aed-m1` … `aed-m11`). One toy task throughout: **reverse a sequence of
digits** (source `[3,7,2,9,5]` -> target `[5,9,2,7,3]`). It is small enough to
train on CPU in seconds, and its ideal attention pattern — a clean
anti-diagonal, source position `j` aligning with target position `T-1-j` — is
something you can actually verify happened, not just believe happened.

Everything runs on **CPU in well under a minute total**. No dataset downloads
— every dataset here is generated in-notebook.

Numbers the modules quote (BLEU clipping example, the variance-growth
argument for scaled dot-product) are re-derived here and checked with `assert`.

| Part | Modules | What runs |
|---|---|---|
| 1 | aed-m1 – aed-m4 | embedding context-independence, encoder/decoder classes, teacher forcing, BLEU from scratch |
| 2 | aed-m5 – aed-m9 | the bottleneck measured directly, attention module, attention-decoder training |
| 3 | aed-m10 | scaled dot-product's variance argument, verified numerically |

## Setup

Colab already ships PyTorch and matplotlib, so the cell below normally
installs nothing. It only fetches a package if it is genuinely missing.

In [ ]:
import importlib.util, subprocess, sys

for pkg, module in [('torch', 'torch'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(module) is None:
        print(f'installing {pkg} …')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    else:
        print(f'{pkg:<12} already present')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import random, math
import matplotlib.pyplot as plt

torch.manual_seed(0); random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
plt.rcParams['figure.figsize'] = (7, 4)

print('torch  ', torch.__version__)
print('device ', DEVICE)

---
# Part 1 — Encoder-Decoder

## aed-m1 · Word embeddings are context-independent

The module's claim, made concrete: `nn.Embedding` returns the *identical*
vector for the same token id, no matter where it sits in the sequence. An RNN
reading that embedding *in context* does not have this problem — same input
id, different position, different hidden state.

In [ ]:
VOCAB = 10          # digits 0-9
embed = nn.Embedding(VOCAB, 8)
rnn = nn.GRU(8, 8, batch_first=True)

# token 3 appears twice, at different positions, in this toy sequence
seq = torch.tensor([[3, 7, 3, 9]])
embedded = embed(seq)

v1, v2 = embedded[0, 0], embedded[0, 2]           # both are embed(3)
print('raw embedding of token 3 at position 0 == at position 2 :', torch.equal(v1, v2))
assert torch.equal(v1, v2)

outputs, _ = rnn(embedded)
h1, h2 = outputs[0, 0], outputs[0, 2]              # RNN hidden state at each position
print('RNN hidden state at position 0 == at position 2         :', torch.equal(h1, h2))
assert not torch.equal(h1, h2)
print('\nSame token id, same raw embedding, but the RNN\'s hidden state differs')
print('because it also depends on everything read before it — context Module 1 promised.')

## aed-m2 / aed-m4 · The task, and the encoder/decoder

The running task for this whole notebook: reverse a sequence of digits.
`<sos>`=10, `<eos>`=11, so the vocabulary is 12 tokens (0-9 plus the two
specials).

In [ ]:
SOS, EOS, PAD = 10, 11, 12
VOCAB_SIZE = 13

def make_batch(batch_size, seq_len):
    """src: random digits. tgt: <sos> + reversed src + <eos>."""
    src = torch.randint(0, 10, (batch_size, seq_len))
    tgt_body = src.flip(dims=[1])
    sos_col = torch.full((batch_size, 1), SOS)
    eos_col = torch.full((batch_size, 1), EOS)
    tgt = torch.cat([sos_col, tgt_body, eos_col], dim=1)
    return src, tgt

src, tgt = make_batch(2, 5)
print('source:', src.tolist())
print('target:', tgt.tolist(), ' <- <sos>=10, reversed source, <eos>=11')

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embed(src)
        outputs, hidden = self.rnn(embedded)     # outputs: every step, hidden: last step only
        return outputs, hidden                    # plain decoder uses only `hidden`

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_tok, hidden):
        embedded = self.embed(input_tok)
        output, hidden = self.rnn(embedded, hidden)
        logits = self.out(output.squeeze(1))
        return logits, hidden

HIDDEN = 32
enc = Encoder(VOCAB_SIZE, 16, HIDDEN).to(DEVICE)
dec = Decoder(VOCAB_SIZE, 16, HIDDEN).to(DEVICE)
print(f'encoder params: {sum(p.numel() for p in enc.parameters()):,}')
print(f'decoder params: {sum(p.numel() for p in dec.parameters()):,}')

## aed-m4 · Teacher forcing, and the training loop

In [ ]:
def train_step_plain(encoder, decoder, src, tgt, optimizer, criterion, teacher_forcing=0.5):
    optimizer.zero_grad()
    _, hidden = encoder(src)
    input_tok = tgt[:, 0:1]
    loss = 0
    for t in range(1, tgt.size(1)):
        logits, hidden = decoder(input_tok, hidden)
        loss = loss + criterion(logits, tgt[:, t])
        use_teacher = random.random() < teacher_forcing
        input_tok = tgt[:, t:t+1] if use_teacher else logits.argmax(-1, keepdim=True)
    loss.backward()
    optimizer.step()
    return loss.item() / (tgt.size(1) - 1)

@torch.no_grad()
def eval_accuracy_plain(encoder, decoder, seq_len, n_batches=20, batch_size=64):
    encoder.eval(); decoder.eval()
    correct = total = 0
    for _ in range(n_batches):
        src, tgt = make_batch(batch_size, seq_len)
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        _, hidden = encoder(src)
        input_tok = tgt[:, 0:1]
        preds = []
        for t in range(1, tgt.size(1) - 1):     # stop before <eos> column
            logits, hidden = decoder(input_tok, hidden)
            input_tok = logits.argmax(-1, keepdim=True)
            preds.append(input_tok)
        preds = torch.cat(preds, dim=1)
        target = tgt[:, 1:-1]
        correct += (preds == target).all(dim=1).sum().item()
        total += batch_size
    encoder.train(); decoder.train()
    return correct / total

optimizer = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=1e-2)
criterion = nn.CrossEntropyLoss()

SEQ_LEN_TRAIN = 5
for step in range(400):
    src, tgt = make_batch(64, SEQ_LEN_TRAIN)
    src, tgt = src.to(DEVICE), tgt.to(DEVICE)
    loss = train_step_plain(enc, dec, src, tgt, optimizer, criterion, teacher_forcing=0.6)
    if step % 100 == 0:
        acc = eval_accuracy_plain(enc, dec, SEQ_LEN_TRAIN)
        print(f'step {step:>4}  loss {loss:.4f}  exact-match acc (len {SEQ_LEN_TRAIN}) {acc:.3f}')

acc_final = eval_accuracy_plain(enc, dec, SEQ_LEN_TRAIN)
print(f'\nfinal accuracy at training length: {acc_final:.3f}')
assert acc_final > 0.7, 'plain encoder-decoder should learn the short reverse task'

## aed-m3 · BLEU from scratch, verified against the module's worked example

In [ ]:
def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def clipped_precision(cand, ref, n):
    cand_grams = ngrams(cand, n)
    if not cand_grams:
        return None
    from collections import Counter
    cand_counts, ref_counts = Counter(cand_grams), Counter(ngrams(ref, n))
    clipped = sum(min(c, ref_counts.get(g, 0)) for g, c in cand_counts.items())
    return clipped / len(cand_grams)

def bleu(cand, ref, max_n=4):
    N = min(max_n, len(cand))
    precisions = [clipped_precision(cand, ref, n) for n in range(1, N + 1)]
    if any(p == 0 for p in precisions):
        return 0.0
    bp = 1.0 if len(cand) >= len(ref) else math.exp(1 - len(ref) / len(cand))
    return bp * math.exp(sum(math.log(p) for p in precisions) / N)

# aed-m3's worked example: same tokens, clipped unigram precision should be 5/7
cand = 'the cat the cat on the mat'.split()
ref  = 'the cat sat on the mat'.split()
p1 = clipped_precision(cand, ref, 1)
print(f'clipped unigram precision: {p1:.4f}  (module worked example: 5/7 = {5/7:.4f})')
assert abs(p1 - 5/7) < 1e-9

print(f'BLEU(exact match)   = {bleu(ref, ref):.4f}')
print(f'BLEU(this candidate) = {bleu(cand, ref):.4f}')
print(f"BLEU(way off)        = {bleu('a dog runs in the park'.split(), ref):.4f}")

---
# Part 2 — Attention

## aed-m5 · The bottleneck, measured directly

Retrain a fresh plain encoder-decoder at a longer sequence length and compare
accuracy against the short-sequence result above. This is the module's claim,
reproduced as a number instead of asserted from a paper.

In [ ]:
def train_plain_at_length(seq_len, steps=400):
    e = Encoder(VOCAB_SIZE, 16, HIDDEN).to(DEVICE)
    d = Decoder(VOCAB_SIZE, 16, HIDDEN).to(DEVICE)
    opt = torch.optim.Adam(list(e.parameters()) + list(d.parameters()), lr=1e-2)
    for _ in range(steps):
        s, t = make_batch(64, seq_len)
        s, t = s.to(DEVICE), t.to(DEVICE)
        train_step_plain(e, d, s, t, opt, criterion, teacher_forcing=0.6)
    return eval_accuracy_plain(e, d, seq_len)

print('plain encoder-decoder, exact-match accuracy vs sequence length:')
plain_results = {}
for L in (3, 6, 10, 15):
    acc = train_plain_at_length(L)
    plain_results[L] = acc
    print(f'  length {L:>2}:  {acc:.3f}')
print('\nAccuracy dropping as length grows is exactly the bottleneck from Module 5 —')
print('the same fixed-size context vector has to carry more information.')

## aed-m6 – aed-m9 · The attention module and decoder

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.U = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs):
        dec_hidden = dec_hidden.unsqueeze(1)
        scores = self.v(torch.tanh(self.W(dec_hidden) + self.U(enc_outputs)))
        weights = torch.softmax(scores.squeeze(-1), dim=1)
        context = torch.bmm(weights.unsqueeze(1), enc_outputs).squeeze(1)
        return context, weights

class AttnDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.attn = BahdanauAttention(hidden_dim)
        self.rnn = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_tok, hidden, enc_outputs):
        embedded = self.embed(input_tok)
        context, weights = self.attn(hidden.squeeze(0), enc_outputs)
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=-1)
        output, hidden = self.rnn(rnn_input, hidden)
        logits = self.out(output.squeeze(1))
        return logits, hidden, weights

def train_step_attn(encoder, decoder, src, tgt, optimizer, criterion, teacher_forcing=0.6):
    optimizer.zero_grad()
    enc_outputs, hidden = encoder(src)
    input_tok = tgt[:, 0:1]
    loss = 0
    for t in range(1, tgt.size(1)):
        logits, hidden, _ = decoder(input_tok, hidden, enc_outputs)
        loss = loss + criterion(logits, tgt[:, t])
        use_teacher = random.random() < teacher_forcing
        input_tok = tgt[:, t:t+1] if use_teacher else logits.argmax(-1, keepdim=True)
    loss.backward()
    optimizer.step()
    return loss.item() / (tgt.size(1) - 1)

@torch.no_grad()
def eval_accuracy_attn(encoder, decoder, seq_len, n_batches=20, batch_size=64):
    encoder.eval(); decoder.eval()
    correct = total = 0
    for _ in range(n_batches):
        src, tgt = make_batch(batch_size, seq_len)
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        enc_outputs, hidden = encoder(src)
        input_tok = tgt[:, 0:1]
        preds = []
        for t in range(1, tgt.size(1) - 1):
            logits, hidden, _ = decoder(input_tok, hidden, enc_outputs)
            input_tok = logits.argmax(-1, keepdim=True)
            preds.append(input_tok)
        preds = torch.cat(preds, dim=1)
        target = tgt[:, 1:-1]
        correct += (preds == target).all(dim=1).sum().item()
        total += batch_size
    encoder.train(); decoder.train()
    return correct / total

def train_attn_at_length(seq_len, steps=400):
    e = Encoder(VOCAB_SIZE, 16, HIDDEN).to(DEVICE)
    d = AttnDecoder(VOCAB_SIZE, 16, HIDDEN).to(DEVICE)
    opt = torch.optim.Adam(list(e.parameters()) + list(d.parameters()), lr=1e-2)
    for _ in range(steps):
        s, t = make_batch(64, seq_len)
        s, t = s.to(DEVICE), t.to(DEVICE)
        train_step_attn(e, d, s, t, opt, criterion, teacher_forcing=0.6)
    return e, d, eval_accuracy_attn(e, d, seq_len)

print('attention encoder-decoder, exact-match accuracy vs sequence length:')
attn_results = {}
attn_models = {}
for L in (3, 6, 10, 15):
    e, d, acc = train_attn_at_length(L)
    attn_results[L] = acc
    attn_models[L] = (e, d)
    print(f'  length {L:>2}:  {acc:.3f}   (plain was {plain_results[L]:.3f})')

## aed-m5 / aed-m6 · The comparison, side by side

In [ ]:
lengths = sorted(plain_results)
plt.plot(lengths, [plain_results[L] for L in lengths], 'o-', label='plain encoder-decoder', color='#b45309')
plt.plot(lengths, [attn_results[L] for L in lengths], 's-', label='attention encoder-decoder', color='#0369a1')
plt.xlabel('sequence length'); plt.ylabel('exact-match accuracy'); plt.ylim(0, 1.05)
plt.legend(); plt.title('The bottleneck (Module 5) vs attention (Module 6), measured'); plt.show()

gap = attn_results[15] - plain_results[15]
print(f'at length 15, attention beats plain by {gap:+.3f} accuracy')
assert gap > -0.05, 'attention should not be meaningfully worse at the longest length tested'

## aed-m7 · The attention weights, visualised

For the reverse task, the ideal alignment is an anti-diagonal: target position
`i` should attend almost entirely to source position `T-1-i`.

In [ ]:
@torch.no_grad()
def attention_matrix(encoder, decoder, src):
    encoder.eval(); decoder.eval()
    enc_outputs, hidden = encoder(src)
    input_tok = torch.tensor([[SOS]], device=DEVICE)
    all_weights = []
    for _ in range(src.size(1)):
        logits, hidden, weights = decoder(input_tok, hidden, enc_outputs)
        all_weights.append(weights.squeeze(0).cpu())
        input_tok = logits.argmax(-1, keepdim=True)
    encoder.train(); decoder.train()
    return torch.stack(all_weights)          # (target_len, source_len)

L = 10
e10, d10 = attn_models[L]
src_sample, _ = make_batch(1, L)
src_sample = src_sample.to(DEVICE)
weights = attention_matrix(e10, d10, src_sample)

plt.imshow(weights.numpy(), cmap='Oranges', aspect='auto')
plt.xlabel('source position'); plt.ylabel('target position (decoding step)')
plt.title(f'attention weights, source = {src_sample[0].tolist()}')
plt.colorbar(label='alpha')
plt.show()

# loose check: the strongest weight per row should trend toward the anti-diagonal
argmax_per_row = weights.argmax(dim=1)
expected = torch.arange(L - 1, -1, -1)
agreement = (argmax_per_row == expected).float().mean().item()
print(f'rows whose strongest attention matches the ideal anti-diagonal: {agreement:.1%}')

---
# Part 3 — The math of attention

## aed-m10 · Why scaled dot-product exists, verified numerically

Module 10's variance argument: for random unit-variance vectors of dimension
`d`, a raw dot product's variance grows with `d`; dividing by `sqrt(d)` keeps
it roughly constant.

In [ ]:
print(f'{"d":>6}{"var(raw dot)":>16}{"var(scaled dot)":>18}')
variances_raw = []
for d in (4, 16, 64, 256, 1024):
    a = torch.randn(20000, d)
    b = torch.randn(20000, d)
    raw = (a * b).sum(dim=1)
    scaled = raw / math.sqrt(d)
    variances_raw.append(raw.var().item())
    print(f'{d:>6}{raw.var().item():>16.2f}{scaled.var().item():>18.4f}')

# variance of the raw dot product should grow roughly linearly with d
ratio = variances_raw[-1] / variances_raw[0]
expected_ratio = 1024 / 4
print(f'\nvar(d=1024) / var(d=4) = {ratio:.1f}   (expected close to {expected_ratio:.0f}, since variance scales with d)')
assert 0.5 * expected_ratio < ratio < 2 * expected_ratio

In [ ]:
# aed-m10's toy 3-dim worked example, reproduced exactly.
H = torch.tensor([[1.00, 0.20, -0.50], [0.30, 0.90, 0.10], [-0.20, 0.40, 0.80]])
s = torch.tensor([0.25, 0.85, 0.05])

raw_scores = H @ s
alpha = torch.softmax(raw_scores, dim=0)
context = (alpha.unsqueeze(1) * H).sum(dim=0)

print('scores e_j :', raw_scores.numpy().round(3))
print('alpha_j    :', alpha.numpy().round(3))
print('context c  :', context.numpy().round(3))
assert abs(alpha.sum().item() - 1.0) < 1e-6

---
## Where to go next

- **Longer sequences, harder tasks.** Push `SEQ_LEN_TRAIN` past 20-25 and even
  the attention model's accuracy will start to soften — the sequential-RNN
  limitation from Module 11, not the representational one from Module 5.
- **Luong-style multiplicative attention.** Swap `BahdanauAttention` for a
  single `nn.Linear` bilinear score (Module 10's "general" score function) and
  compare training speed — fewer parameters, one matmul instead of a small MLP.
- **Self-attention.** The natural next question this track raises but doesn't
  answer: what if tokens attended to each other *within* the same sequence,
  not just decoder-to-encoder — and the recurrence was dropped entirely? That
  is the Transformer, and a different track's starting point.

Re-run with a bigger `HIDDEN` and more `steps` and the accuracy numbers above
improve — but the *shape* of the plain-vs-attention gap, and the anti-diagonal
attention pattern, will not change.